# Mean Surface Clusters
The idea here is to take some mean of surface values (or possibly subset of) to cluster on -- the hope is that this can give insights into water masses on the shelf as well as in the basins (which is all the current method is realistically doing)

In [ ]:
import pandas as pd
import xarray as xr
import numpy as np

import matplotlib as mpl
import matplotlib.pyplot as plt

import cartopy.crs as ccrs
import cartopy.feature as cfeature

In [ ]:
def geoaxes(n_rows=1, n_cols=1, **mpl_kw):
    '''
    Creates a set of GeoAxes (on which maps etc can be plotted)

    Args:
        n_rows (int): Number of rows. Default = 1
        n_cols (int): Number of columns. Default = 1

    Returns:
        fig, ax(s) 
    '''

    assert n_rows >= 1
    assert n_cols >= 1

    fig, axs = plt.subplots(n_rows, n_cols, **mpl_kw, subplot_kw={'projection': ccrs.PlateCarree(central_longitude=180)})

    if n_rows + n_cols > 2:
        axs = axs.ravel()

    if n_rows == 1 and n_cols == 1:
        axs.add_feature(cfeature.COASTLINE, alpha=0.2)
        axs.add_feature(cfeature.BORDERS, alpha=0.2)
        axs.add_feature(cfeature.LAND, alpha=0.2)
    
    return fig, axs

# techincally GeoAxes is cartopy.mpl.geoaxes.GeoAxes
def plot_all_vals(ds, ax=None, color=None, LON_VAR='LONGITUDE', LAT_VAR='LATITUDE', LON_SHIFT = 180, ):
    '''
    Plots all points (latitude-longitude) from a dataset; gives a general gist of where the data lies. 

    Args:
        ds (xr.Dataset): Dataset of values
        ax (GeoAxes; optional): Axis to plot data on
        color (string; optional): Variable to color by

    Returns:
    
    '''

    if ax == None:
        fig, ax = geoaxes()

    lons = ds[LON_VAR] + LON_SHIFT
    lats = ds[LAT_VAR]

    ax.scatter(
        lons, lats, c=color, s=1
    )

In [ ]:
data_path = './data/ACOD_CTD_v1.0.nc'

data = xr.open_dataset(data_path)

## NOTE: according to the paper accompanying the dataset, the PPT will differ from PSU by '0.01-0.05' across the dataset; for our purposes this is fine, but still worth noting nonetheless
## All cruises pre-August 1995 were in PPT (which is around 14000 profiles so nearly half); repeating with cleaned data shows around half the data is from before 1995 too.
data['Merged_Salinity'] = data.Salinity_PPT_with_QC_applied.combine_first(data.Salinity_PSS_with_QC_applied)

data['Region_Name'] = data.REGION.sum(axis=0)

data_reduced = data[['Temperature', 'Merged_Salinity', 'BOTTOM_DEPTH']]
data_info = data[['BOTTOM_DEPTH', 'Region_Name', 'CRUISE_NUMBER', 'CRUISE_NAME']]

interpolated_data = data_reduced.interpolate_na(dim='PRESSURE')

# 'Chukchi Sea', 'Gulf of Alaska', 'Bering Sea'
region_choice = '' #'Chukchi Sea'
regional_data = interpolated_data.where(data_info.Region_Name != f'{region_choice:<14}')

In [ ]:
plot_all_vals(regional_data)